# 45 · GSE69529 · bulk_RNA_seq · healthy children, the reference

Reads the authors' count files and the series matrix in `data/GSE69529/`. Writes `healthy_counts.rds` and
`healthy_samples.rds` in `data/run_artifacts/CANDLE_SAVI/`.

**The data.** GEO GSE69529 (DeBerg HA, Zaidi MB, Altman MC, et al. Shared and organism-specific host responses
to childhood diarrheal diseases revealed by whole blood transcript profiling. *PLoS One* 2018;13:e0192082):
whole blood RNA sequencing of children in Mexico with diarrhea and of control children. The authors deposited
their read counts per Ensembl gene (htseq-count, GRCh38) in two files, one per sequencing set. The RNA was
extracted with the MagMax kit for stabilized blood tubes and globin-reduced with GLOBINclear before sequencing
(GEO protocol).

The two files:
- `mexico_processed_data_for_GEO.csv`: counts after the authors' sample quality filter, with non-protein-coding
  and mitochondrial genes removed;
- `mexico2_data.csv`: counts before any gene filter.

**Use here.** The control children are the healthy reference for notebooks 46, 46b and 47. They were sequenced
in a different lab from the CANDLE, SAVI, AGS, UAID and SLE samples, from different blood tubes; a gene that
differs between them and a disease may differ because of the disease or because of the lab.

**Healthy blocks.** GEO records age only as under 2 years or 2 years or older; the two are kept as separate
healthy blocks.

## Sample information

In [1]:
source("../src/paths.R")
suppressMessages(library(GEOquery))
pd <- pData(getGEO(filename = raw("GSE69529", "GSE69529_series_matrix.txt.gz"), getGPL = FALSE))
pd <- data.frame(title = pd$title, gsm = pd$geo_accession, group = pd[["group:ch1"]], age = pd[["age:ch1"]],
                 source = pd$source_name_ch1, row.names = NULL)
table(group = pd$group, age = pd$age)

          age
group      2 year-old or older <2 year-old
  Control                   24          23
  Diarrhea                  99         109

**Explanation**
- `getGEO()` and `pData()` as in notebook 01; `pd` keeps each sample's title (the code used in the count files),
  GEO accession, group (Control or Diarrhea), age band and source description.
- The table crosses group with age band.

**Result.** 255 samples: 47 control (23 under 2, 24 aged 2 or older) and 208 diarrhea.

## Read the two count files

In [2]:
read_counts <- function(f, id_col, first_sample) {
  x <- read.csv(raw("GSE69529", f), check.names = FALSE)
  m <- as.matrix(x[, first_sample:ncol(x)]); rownames(m) <- x[[id_col]]
  m
}
set1 <- read_counts("GSE69529_mexico_processed_data_for_GEO.csv.gz", "Ensembl Gene ID", 7)
set2 <- read_counts("GSE69529_GSM2241808-GSM2241855_mexico2_data.csv.gz", "Ensembl.Gene.ID", 3)
data.frame(set = c("set1", "set2"), genes = c(nrow(set1), nrow(set2)), samples = c(ncol(set1), ncol(set2)),
           repeated_gene_ids = c(sum(duplicated(rownames(set1))), sum(duplicated(rownames(set2)))),
           not_whole = c(sum(set1 != round(set1)), sum(set2 != round(set2))), negative = c(sum(set1 < 0), sum(set2 < 0)))

set,genes,samples,repeated_gene_ids,not_whole,negative
<chr>,<int>,<int>,<int>,<int>,<int>
set1,20347,203,0,0,0
set2,20980,48,0,0,0


**Explanation**
- `read_counts(f, id_col, first_sample)` reads one file with `read.csv()` and keeps the count columns, from
  `first_sample` on, as a matrix with the Ensembl gene identifiers as row names. `check.names = FALSE` keeps the
  column names as written, for example `CT026, repeat`.
- In the first file the counts start in column 7 (after gene ID, transcript ID, symbol, start, end and
  chromosome); in the second in column 3 (after gene ID and symbol).
- The table gives genes and samples per file, and checks for repeated gene identifiers, counts that are not whole
  numbers and negative counts.

**Result.** Set 1: 20,347 genes x 203 samples; set 2: 20,980 genes x 48 samples. No repeated gene identifier; every
count is a whole number of 0 or more.

## The control children, one library each

In [3]:
ctrl <- pd[pd$group == "Control", ]
in1 <- intersect(colnames(set1), ctrl$title); in2 <- intersect(colnames(set2), ctrl$title)
child <- function(x) sub(",\\s*repeat$", "", x)
keep2 <- in2[!child(in2) %in% child(in1)]
c(controls_in_set1 = length(in1), controls_in_set2 = length(in2),
  set2_libraries_of_children_already_in_set1 = length(in2) - length(keep2), children = length(in1) + length(keep2))

controls_in_set1 
                                        24 
                          controls_in_set2 
                                        23 
set2_libraries_of_children_already_in_set1 
                                         4 
                                  children 
                                        43

**Explanation**
- `ctrl` keeps the control samples; `in1` and `in2` the control columns of each file.
- `child()` removes the word `repeat` from a column name, so that a repeated library is matched to its child:
  `child("CT026, repeat")` gives `"CT026"`.
- `keep2` keeps the set-2 controls whose child is not already in set 1: each child keeps one library, the one in
  set 1.

**Result.** 24 control libraries in set 1 and 23 in set 2; 4 of the set-2 libraries repeat children of set 1
(CT026, CT030, CT056, CT057) and are dropped. 43 children remain, one library each.

## Genes in both sets

In [4]:
genes <- intersect(rownames(set1), rownames(set2))
counts <- cbind(set1[genes, in1], set2[genes, keep2])
healthy <- ctrl[match(colnames(counts), ctrl$title), ]
healthy$set <- ifelse(healthy$title %in% in1, "set1", "set2")
healthy$block <- ifelse(healthy$age == "<2 year-old", "healthy_under2", "healthy_2plus")
healthy$sample <- sprintf("H%03d", seq_len(nrow(healthy)))
colnames(counts) <- healthy$sample
c(genes_set1 = nrow(set1), genes_set2 = nrow(set2), genes_in_both = length(genes))
table(block = healthy$block, set = healthy$set)

genes_set1    genes_set2 genes_in_both 
        20347         20980         15564

                set
block            set1 set2
  healthy_2plus    13    9
  healthy_under2   11   10

**Explanation**
- `intersect()` keeps the genes in both files. The first file holds protein-coding genes only, so the genes in
  both are protein-coding genes.
- `counts` puts the chosen control columns side by side; `healthy` holds their sample information in the same
  order.
- `set` records which file each child comes from, `block` the healthy block by age, and `sample` gives each child
  a code, `H001` on, used in later notebooks and figures.
- The table counts children by healthy block and set.

**Result.** 15,564 genes are in both files. Healthy blocks: 21 children under 2 (11 in set 1, 10 in set 2) and 22
aged 2 or older (13 and 9).

## Save

In [5]:
saveRDS(counts, art("CANDLE_SAVI", "healthy_counts.rds"))
saveRDS(healthy[, c("sample", "title", "gsm", "set", "block", "age", "source")], art("CANDLE_SAVI", "healthy_samples.rds"))
dim(counts)

[1] 15564    43

**Explanation**
- `healthy_counts.rds` holds the counts of the healthy children, genes by children; `healthy_samples.rds` their
  sample information, with the GEO codes and accessions (public identifiers).

## References

1. DeBerg HA, Zaidi MB, Altman MC, Khaenam P, Gersuk VH, Campos FD, et al. Shared and organism-specific host
   responses to childhood diarrheal diseases revealed by whole blood transcript profiling. *PLoS One*
   2018;13:e0192082.
2. Davis S, Meltzer PS. GEOquery: a bridge between the Gene Expression Omnibus (GEO) and BioConductor.
   *Bioinformatics* 2007;23:1846-1847.
3. R Core Team. *R: A Language and Environment for Statistical Computing*. R Foundation for Statistical
   Computing, Vienna, Austria.